# Multinomial models

This chapter covers the `Multinomial` family for $K$-class classification,
its Kronecker-product design that shares base learners across classes, and the
softmax link. Read it after [](families.ipynb) (Binomial baseline, loss/gradient
structure) and before [](gamlss_framework.ipynb) (which extends the same
multi-predictor idea to distributional parameters).

The `Multinomial` family generalizes the Binomial to $K > 2$ classes. With class
$K$ as reference, it models $K-1$ logit predictors and shares base learners across
classes through a **Kronecker-product design**.

## The model

For observation $i$ with class-$k$ logit $f_{ik}$ ($k = 1,\dots,K-1$; the
reference class has logit 0),

$$
P(y_i = k) = \frac{e^{f_{ik}}}{1 + \sum_{l=1}^{K-1} e^{f_{il}}}, \qquad
P(y_i = K) = \frac{1}{1 + \sum_{l} e^{f_{il}}} .
$$

With the dummy-coded response $y_{ik} = \mathbf{1}\{y_i = k\}$ (the reference column
dropped), the loss and gradient are

$$
\rho_i = -\sum_{k=1}^{K-1} y_{ik}\,\log P(y_i=k), \qquad u_{ik} = y_{ik} - P(y_i=k).
$$

```{note}
Numerically the softmax is max-stabilized: subtract $m_i = \max(0, \max_k f_{ik})$
(the 0 accounts for the reference class) before exponentiating. Use $e^{-m_i}$,
not 1, as the reference term; otherwise the normalization is wrong when
$m_i > 0$.
```

## Sharing learners across classes: the Kronecker design

A covariate effect is shared across the $K-1$ class predictors via a row-wise
Kronecker product. If a base learner has marginal design $X_1$ (the covariate part,
$n \times p_1$) and the class structure is $X_2 = I_{K-1}$, the combined design is
$X_2 \otimes X_1$ and the response is the stacked $n(K-1)$ vector. The penalized
fit decouples per class:

$$
\hat\beta_k = (X_1^\top W X_1 + P)^{-1} X_1^\top W\, u_{\cdot k}, \qquad k=1,\dots,K-1,
$$

so a single solve factorization serves all classes. The penalty $P$ determines
the smoothness sharing; see the next section.

## Penalty for smooth multinomial effects

For a linear (`bols`) effect $P = 0$ (or a small ridge). For a **smooth** (`bbs`)
effect, R `mboost`'s `%O%` operator forms the anisotropic Kronecker penalty

$$
P = \lambda\,\big[\,K_2 \otimes I_{p_1} + I_{p_2} \otimes K_1\,\big],
$$

where $K_1$ is the bbs difference penalty and $K_2$ is the class learner's penalty.
With the standard class learner `bols(class, df = K-1, contr.dummy)`, $K_2 = I$,
so the per-class penalty becomes $\lambda\,(I + K_1)$ — the bbs difference penalty
plus a ridge identity — and the combined degrees of freedom target is
$\mathrm{df}_1 \cdot \mathrm{df}_2$, split over the $K-1$ identical class blocks
(so the per-class target is $\mathrm{df}_1$). `mboostrs` reproduces this exactly
(verified to $\sim 10^{-11}$); the detailed reconstruction is in
[](../implementation/kronecker_multinomial).

## Worked example: 3 classes on 4 observations

Consider $K = 3$ classes with reference class C, $n = 4$ observations,
and a single covariate $x$ (so $X_1 = [1, x]$, $p_1 = 2$). The logits
are initially at their offset values $f_{i1} = f_{i2} = 0$.

| $i$ | $y_i$ | $x_i$ |
|-----|--------|-------|
| 1   | A      | 1.0   |
| 2   | B      | 2.0   |
| 3   | C      | 0.5   |
| 4   | A      | 3.0   |

**Dummy-coded response** ($K-1 = 2$ columns, reference C dropped):

$$
y = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \\ 1 & 0 \end{pmatrix}
\quad\text{stacked as}\quad
y_{\text{vec}} = (1, 0, 0, 1, 0, 0, 0, 0).
$$

**Softmax at $f = 0$:** $P(y = k) = 1/3$ for all classes (uniform).
The per-observation gradient for class $k$ is $u_{ik} = y_{ik} - P(y_i = k)$:

$$
U = \begin{pmatrix} 2/3 & -1/3 \\ -1/3 & 2/3 \\ -1/3 & -1/3 \\ 2/3 & -1/3 \end{pmatrix}
\quad\text{stacked as}\quad
u_{\text{vec}} = (\tfrac{2}{3}, -\tfrac{1}{3}, -\tfrac{1}{3}, \tfrac{2}{3},\;
-\tfrac{1}{3}, \tfrac{2}{3}, -\tfrac{1}{3}, -\tfrac{1}{3}).
$$

**Kronecker design.** With $X_1 = \begin{pmatrix}1 & 1\\1 & 2\\1 & 0.5\\1 & 3\end{pmatrix}$
and $X_2 = I_2$:

$$
X_{\text{kronecker}} = I_2 \otimes X_1 =
\begin{pmatrix}
  1 & 1 & 0 & 0 \\
  1 & 2 & 0 & 0 \\
  1 & 0.5 & 0 & 0 \\
  1 & 3 & 0 & 0 \\
  0 & 0 & 1 & 1 \\
  0 & 0 & 1 & 2 \\
  0 & 0 & 1 & 0.5 \\
  0 & 0 & 1 & 3 \\
\end{pmatrix}.
$$

Columns 1–2 are the intercept+slope for class A; columns 3–4 for class B.
The block-diagonal structure means the normal equations decouple: class
A's fit uses columns 1–2 against the first 4 elements of $u_{\text{vec}}$;
class B's uses columns 3–4 against the last 4. One factorization of $X_1^\top X_1$ serves both classes, reducing the
work required for the two class-specific fits.

**Per-class fit (linear, $P = 0$):**

$$
\hat\beta_A = (X_1^\top X_1)^{-1} X_1^\top u_{\cdot A},
\qquad
\hat\beta_B = (X_1^\top X_1)^{-1} X_1^\top u_{\cdot B}.
$$

Both solves reuse $(X_1^\top X_1)^{-1}$. This shared factorization is the
main computational benefit of the Kronecker construction.

In [1]:
import numpy as np
from mboostrs._rust import family_diagnostics

# Reproduce the worked example above at f=0 (interleaved layout: the engine
# stores y and f as [y_1A, y_1B, y_2A, y_2B, ...], one observation's K-1
# values contiguous).
y = np.array([1, 0,  0, 1,  0, 0,  1, 0], dtype=float)  # A, B, C, A
f = np.zeros(8)
w = np.ones(4)  # one weight per observation, not per class column

_, grad, risk = family_diagnostics("multinomial", {"n_classes": 3.0}, y, f, w)
U = grad.reshape(4, 2)
print("U =\n", U)
print(f"risk = {risk:.6f}  (3 log 3 = {3 * np.log(3):.6f})")

U_expected = np.array([[2/3, -1/3], [-1/3, 2/3], [-1/3, -1/3], [2/3, -1/3]])
np.testing.assert_allclose(U, U_expected, atol=1e-10)
np.testing.assert_allclose(risk, 3 * np.log(3), atol=1e-8)


U =
 [[ 0.66666667 -0.33333333]
 [-0.33333333  0.66666667]
 [-0.33333333 -0.33333333]
 [ 0.66666667 -0.33333333]]
risk = 3.295837  (3 log 3 = 3.295837)


The engine's gradient matrix and risk match the hand-derived values exactly.
Observation 3 (the reference class C) contributes 0 to the risk — the loss
sums $-\log P(y_i=k)$ only over the $K-1$ non-reference indicator columns,
so a reference-class observation's own correctly-assigned probability never
appears as a term. This implementation follows the displayed expression and reproduces R
`mboost`'s `Multinomial()` risk exactly. It differs from the full multinomial
negative log-likelihood, which includes the reference-class probability.

## Softmax probability calculation

The following calculation applies the softmax directly to the fitted model's
stored $K-1$ logits and compares the probabilities with
`predict(type="response")`. Agreement verifies that prediction uses the same
softmax model after the Kronecker learner solves.

In [2]:
import numpy as np
import mboostrs as mb

rng = np.random.default_rng(6)
n = 300
x = rng.uniform(-3, 3, n)
eta = np.column_stack([np.sin(x), -0.5 * x])
p_true = np.exp(np.column_stack([eta, np.zeros(n)]))
p_true /= p_true.sum(1, keepdims=True)
y = np.array(["A", "B", "C"])[np.array([rng.choice(3, p=pi) for pi in p_true])]
data = {"x": x, "y": y}

m = mb.gamboost("y ~ bbs(x, df=4)", data=data, family=mb.Multinomial(),
                control=mb.boost_control(mstop=300, nu=0.1))
link = m.predict(type="link")            # (n, K-1) logits, reference implicit 0
prob_engine = m.predict(type="response")  # (n, K)

# Direct softmax: append the reference class's logit of 0, max-stabilize
# the same way the family's negative_gradient does, and normalize.
full_logit = np.column_stack([link, np.zeros(n)])
mstab = np.maximum(0.0, full_logit.max(axis=1, keepdims=True))
prob_direct = np.exp(full_logit - mstab)
prob_direct /= prob_direct.sum(axis=1, keepdims=True)

print(f"max |direct - engine| = {np.abs(prob_direct - prob_engine).max():.2e}")
print(f"row sums: min={prob_engine.sum(1).min():.6f}  max={prob_engine.sum(1).max():.6f}")

np.testing.assert_allclose(prob_direct, prob_engine, atol=1e-8)
np.testing.assert_allclose(prob_engine.sum(axis=1), 1.0, atol=1e-10)


max |direct - engine| = 2.22e-16
row sums: min=1.000000  max=1.000000


The two ways of computing class probabilities agree to floating-point
precision ($< 10^{-15}$), and every row of `predict(type="response")` sums to
1 to $10^{-10}$. The Kronecker design only changes how the $K-1$ base-learner
fits are solved; the predictive model is an ordinary softmax over those fits.



## Supported learners

Only linear (`bols`) and smooth (`bbs`) effects are available; other learner
types are rejected. Prediction returns the
$(n, K)$ probability matrix (reference class last), in-sample and on new data.
AIC is unavailable: the Kronecker learner has no hat matrix, and R cannot
compute AIC for the `%O%` model either.

## Summary

- The multinomial response is dummy-coded into $K-1$ columns (reference
  dropped) and stacked row-major. The softmax, loss, and gradient operate
  on the expanded vector.
- Shared effects use the Kronecker construction $X_2 \otimes X_1$ where
  $X_1$ is the marginal design and $X_2 = I_{K-1}$. The block-diagonal
  normal equations give $K-1$ independent solves sharing one factorization.
- For smooth effects, the per-class penalty is $\lambda(I + K_1)$ — the
  `bols`-class ridge is added to the smooth penalty. Only `bols` and `bbs`
  are Kronecker-wrappable.

## See also

- [](../usage/multinomial) — runnable code.
- [](../implementation/kronecker_multinomial) — the Kronecker-product
  design construction and the `+I` penalty-leak reconstruction.